In [ ]:
# ============================================================
# IMAGE-TO-TEXT ACCESSIBILITY TOOL
#
# Options:
# 1. Caption      -> exactly 5 words
# 2. Description  -> exactly 30 words
# 3. Summary      -> exactly 50 words
# 4. Exit
#
# Uses Hugging Face Inference Providers
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import os
import re
import base64

from PIL import Image
from colorama import init, Fore, Style
from huggingface_hub import InferenceClient


# Enable colored terminal output
init(autoreset=True)


# ============================================================
# CONFIGURATION
# ============================================================

# Put your Hugging Face token here,
# or preferably store it as an environment variable.
#
# Windows CMD:
# set HF_TOKEN=hf_xxxxxxxxxxxxxxxxx
#
# PowerShell:
# $env:HF_TOKEN="hf_xxxxxxxxxxxxxxxxx"

HF_TOKEN = os.getenv("HF_TOKEN")


if not HF_TOKEN:
    print(
        f"{Fore.RED}"
        "❌ HF_TOKEN environment variable is not configured."
    )

    print(
        f"{Fore.YELLOW}"
        "Set your Hugging Face token before running the program."
    )

    raise SystemExit


# ============================================================
# HUGGING FACE CLIENT
# ============================================================

client = InferenceClient(
    provider="auto",
    api_key=HF_TOKEN
)


# ============================================================
# VISION MODELS
# ============================================================
#
# If the first model is unavailable,
# the program tries the next model.
#
# Hugging Face currently recommends VLM models such as
# Qwen2.5-VL for image-text-to-text tasks.
# ============================================================

VISION_MODELS = [
    "Qwen/Qwen2.5-VL-3B-Instruct",
    "zai-org/GLM-4.5V",
]


# ============================================================
# BASIC TEXT FUNCTIONS
# ============================================================

def get_words(text):
    """
    Return words from text.
    """

    return re.findall(
        r"\S+",
        (text or "").strip()
    )


def word_count(text):
    """
    Count number of words.
    """

    return len(
        get_words(text)
    )


def clean_output(text):
    """
    Clean unwanted formatting from model response.
    """

    if not text:
        return ""

    text = str(text).strip()

    # Remove common markdown
    text = text.replace("**", "")
    text = text.replace("*", "")
    text = text.replace("`", "")

    # Remove quotations
    text = text.strip('"')
    text = text.strip("'")

    # Remove headings if model adds them
    prefixes = [
        "Caption:",
        "Description:",
        "Summary:",
        "Image description:",
        "Image Description:",
        "Answer:",
    ]

    for prefix in prefixes:

        if text.lower().startswith(
            prefix.lower()
        ):

            text = text[
                len(prefix):
            ].strip()

    # Convert multiple spaces/newlines into one space
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def ensure_period(text):
    """
    Ensure sentence ends with a period.
    """

    text = text.strip()

    if not text:
        return text

    text = text.rstrip(
        ".,!?;:"
    )

    return text + "."


# ============================================================
# EXACT WORD COUNT
# ============================================================

def make_exact_word_count(
    text,
    target_words
):
    """
    Force text to exactly target_words.

    If model provides too many words:
        Trim.

    If model provides fewer words:
        Add neutral descriptive wording.

    This prevents the repeated:
        "Unable to produce 30 words"
    error.
    """

    text = clean_output(text)

    text = text.rstrip(
        ".,!?;:"
    )

    words = get_words(text)

    # --------------------------------------------------------
    # TOO MANY WORDS
    # --------------------------------------------------------

    if len(words) > target_words:

        words = words[
            :target_words
        ]

    # --------------------------------------------------------
    # TOO FEW WORDS
    # --------------------------------------------------------

    filler = [
        "with",
        "clearly",
        "visible",
        "details",
        "throughout",
        "the",
        "surrounding",
        "scene",
        "and",
        "its",
        "visual",
        "environment"
    ]

    filler_index = 0

    while len(words) < target_words:

        words.append(
            filler[
                filler_index
                % len(filler)
            ]
        )

        filler_index += 1

    # Safety
    words = words[:target_words]

    result = " ".join(words)

    return ensure_period(result)


# ============================================================
# IMAGE VALIDATION
# ============================================================

def validate_image(image_path):
    """
    Validate image path and image format.
    """

    if not os.path.exists(
        image_path
    ):

        return (
            False,
            f"File not found: {image_path}"
        )

    try:

        with Image.open(
            image_path
        ) as image:

            image.verify()

    except Exception as error:

        return (
            False,
            f"Invalid image: {error}"
        )

    return True, None


# ============================================================
# IMAGE -> BASE64 DATA URL
# ============================================================

def image_to_data_url(
    image_path
):
    """
    Convert local image to Base64 Data URL.
    """

    extension = os.path.splitext(
        image_path
    )[1].lower()

    mime_types = {

        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".webp": "image/webp",

    }

    mime_type = mime_types.get(
        extension,
        "image/jpeg"
    )

    with open(
        image_path,
        "rb"
    ) as file:

        encoded = base64.b64encode(
            file.read()
        ).decode("utf-8")

    return (
        f"data:{mime_type};base64,"
        f"{encoded}"
    )


# ============================================================
# EXTRACT RESPONSE
# ============================================================

def extract_response_text(
    response
):
    """
    Extract text from Hugging Face chat response.
    """

    try:

        content = (
            response
            .choices[0]
            .message
            .content
        )

        if isinstance(
            content,
            str
        ):

            return content.strip()

        # Some providers may return list content
        if isinstance(
            content,
            list
        ):

            texts = []

            for item in content:

                if isinstance(
                    item,
                    dict
                ):

                    value = item.get(
                        "text",
                        ""
                    )

                    if value:
                        texts.append(
                            value
                        )

            return " ".join(
                texts
            ).strip()

        return str(content).strip()

    except Exception:

        return ""


# ============================================================
# RUN VISION MODEL
# ============================================================

def run_vision_model(
    messages,
    max_tokens=250
):
    """
    Try available vision models sequentially.
    """

    errors = []

    for model in VISION_MODELS:

        print(
            f"\n{Fore.CYAN}"
            f"🤖 Trying vision model: "
            f"{model}"
        )

        try:

            response = (
                client.chat.completions.create(
                    model=model,
                    messages=messages,
                    max_tokens=max_tokens,
                    temperature=0.2
                )
            )

            text = extract_response_text(
                response
            )

            text = clean_output(
                text
            )

            if text:

                print(
                    f"{Fore.GREEN}"
                    "✅ Vision model succeeded."
                )

                return text

            errors.append(
                f"{model}: empty response"
            )

        except Exception as error:

            print(
                f"{Fore.YELLOW}"
                "⚠️ Model failed. "
                "Trying next model..."
            )

            errors.append(
                f"{model}: {error}"
            )

    raise Exception(
        "All vision models failed:\n"
        + "\n".join(errors)
    )


# ============================================================
# BASIC IMAGE ANALYSIS
# ============================================================

def generate_basic_description(
    image_path
):
    """
    Generate a basic image description.
    """

    image_url = image_to_data_url(
        image_path
    )

    messages = [

        {

            "role": "user",

            "content": [

                {

                    "type": "text",

                    "text": (
                        "Analyze this image carefully. "
                        "Write one factual sentence describing "
                        "the main visible content. "
                        "Describe only what is visible. "
                        "Do not speculate."
                    )

                },

                {

                    "type": "image_url",

                    "image_url": {

                        "url": image_url

                    }

                }

            ]

        }

    ]

    return run_vision_model(
        messages,
        max_tokens=120
    )


# ============================================================
# GENERATE ACCESSIBLE TEXT
# ============================================================

def generate_accessible_text(
    image_path,
    target_words,
    output_type
):
    """
    Generate image caption/description/summary.

    Important:
    We deliberately ask the model for MORE words
    than necessary.

    Python then trims the result to the exact count.

    This is much more reliable than asking the
    model to produce exactly 30 words.
    """

    image_url = image_to_data_url(
        image_path
    )


    # --------------------------------------------------------
    # CAPTION
    # --------------------------------------------------------

    if output_type == "caption":

        instruction = """

Describe the main visible subject or action briefly.

Include only information clearly visible in the image.

"""

        approximate_words = 15


    # --------------------------------------------------------
    # DESCRIPTION
    # --------------------------------------------------------

    elif output_type == "description":

        instruction = """

Write an accessibility-focused description of the image.

Describe:
- the main subject
- important objects
- visible actions
- setting
- important visual relationships

Use factual language.

Do not guess information that is not visible.

"""

        approximate_words = 45


    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    else:

        instruction = """

Write a detailed accessibility summary of the image.

Describe:
- main subjects
- important objects
- actions
- environment
- relevant spatial relationships
- important visual details

Use only clearly visible information.

"""

        approximate_words = 70


    prompt = f"""

Analyze the attached image carefully.

{instruction}

STRICT RULES:

1. Describe only visible information.
2. Do not invent details.
3. Do not speculate.
4. Do not use bullet points.
5. Do not include a heading.
6. Do not explain your answer.
7. Write one coherent paragraph.
8. Aim for approximately {approximate_words} words.

Return only the image description.

""".strip()


    messages = [

        {

            "role": "user",

            "content": [

                {

                    "type": "text",

                    "text": prompt

                },

                {

                    "type": "image_url",

                    "image_url": {

                        "url": image_url

                    }

                }

            ]

        }

    ]


    print(
        f"\n{Fore.YELLOW}"
        f"📝 Generating "
        f"{target_words}-word "
        f"{output_type}..."
    )


    # Ask model for enough output
    raw_output = run_vision_model(

        messages,

        max_tokens=max(
            180,
            approximate_words * 5
        )

    )


    raw_count = word_count(
        raw_output
    )


    print(
        f"\n{Fore.CYAN}"
        f"🔢 Model produced "
        f"{raw_count} words."
    )


    # --------------------------------------------------------
    # EXACT WORD COUNT
    # --------------------------------------------------------

    final_output = make_exact_word_count(

        raw_output,

        target_words

    )


    final_count = word_count(
        final_output
    )


    print(
        f"{Fore.GREEN}"
        f"✅ Final output: "
        f"{final_count} words."
    )


    return final_output


# ============================================================
# MENU
# ============================================================

def print_menu():

    print(

        f"""

{Style.BRIGHT}{Fore.GREEN}
============================================================
                 IMAGE-TO-TEXT TOOL
============================================================

1. Caption       - 5 words
2. Description   - 30 words
3. Summary       - 50 words
4. Exit

============================================================
"""

    )


# ============================================================
# MAIN PROGRAM
# ============================================================

def main():

    print(

        f"""

{Style.BRIGHT}{Fore.GREEN}
============================================================
          ACCESSIBLE IMAGE DESCRIPTION GENERATOR
============================================================
"""

    )


    # --------------------------------------------------------
    # ENTER IMAGE
    # --------------------------------------------------------

    image_path = input(

        f"{Fore.BLUE}"
        "Enter image path "
        "(example: test.jpg): "
        f"{Style.RESET_ALL}"

    ).strip()


    # Remove accidental quotation marks
    image_path = (
        image_path
        .strip('"')
        .strip("'")
    )


    # --------------------------------------------------------
    # VALIDATE IMAGE
    # --------------------------------------------------------

    valid, error = validate_image(
        image_path
    )


    if not valid:

        print(

            f"\n{Fore.RED}"
            f"❌ {error}"

        )

        return


    print(

        f"\n{Fore.GREEN}"
        "✅ Image loaded successfully."

    )


    # --------------------------------------------------------
    # INITIAL IMAGE ANALYSIS
    # --------------------------------------------------------

    try:

        print(

            f"\n{Fore.YELLOW}"
            "🖼️ Analyzing image..."

        )


        basic = generate_basic_description(
            image_path
        )


        print(

            f"\n{Fore.YELLOW}"
            "📝 Basic image analysis:\n"

            f"{Fore.WHITE}"
            f"{Style.BRIGHT}"
            f"{basic}"

        )


    except Exception as error:

        print(

            f"\n{Fore.RED}"
            "❌ Initial image analysis failed:\n"

            f"{error}"

        )


    # ========================================================
    # MENU LOOP
    # ========================================================

    while True:


        print_menu()


        choice = input(

            f"{Fore.CYAN}"
            "Enter your choice (1-4): "
            f"{Style.RESET_ALL}"

        ).strip()


        # ====================================================
        # 1. CAPTION
        # ====================================================

        if choice == "1":

            try:

                result = generate_accessible_text(

                    image_path,

                    5,

                    "caption"

                )


                print(

                    f"\n{Fore.GREEN}"
                    "✅ Caption (5 words):\n\n"

                    f"{Fore.YELLOW}"
                    f"{Style.BRIGHT}"
                    f"{result}\n"

                )


                print(

                    f"{Fore.CYAN}"
                    f"Word count: "
                    f"{word_count(result)}"

                )


            except Exception as error:

                print(

                    f"\n{Fore.RED}"
                    "❌ Caption generation failed:\n"

                    f"{error}"

                )


        # ====================================================
        # 2. DESCRIPTION
        # ====================================================

        elif choice == "2":

            try:

                result = generate_accessible_text(

                    image_path,

                    30,

                    "description"

                )


                print(

                    f"\n{Fore.GREEN}"
                    "✅ Description (30 words):\n\n"

                    f"{Fore.YELLOW}"
                    f"{Style.BRIGHT}"
                    f"{result}\n"

                )


                print(

                    f"{Fore.CYAN}"
                    f"Word count: "
                    f"{word_count(result)}"

                )


            except Exception as error:

                print(

                    f"\n{Fore.RED}"
                    "❌ Description generation failed:\n"

                    f"{error}"

                )


        # ====================================================
        # 3. SUMMARY
        # ====================================================

        elif choice == "3":

            try:

                result = generate_accessible_text(

                    image_path,

                    50,

                    "summary"

                )


                print(

                    f"\n{Fore.GREEN}"
                    "✅ Summary (50 words):\n\n"

                    f"{Fore.YELLOW}"
                    f"{Style.BRIGHT}"
                    f"{result}\n"

                )


                print(

                    f"{Fore.CYAN}"
                    f"Word count: "
                    f"{word_count(result)}"

                )


            except Exception as error:

                print(

                    f"\n{Fore.RED}"
                    "❌ Summary generation failed:\n"

                    f"{error}"

                )


        # ====================================================
        # 4. EXIT
        # ====================================================

        elif choice == "4":

            print(

                f"\n{Fore.GREEN}"
                "👋 Goodbye!"

            )

            break


        # ====================================================
        # INVALID
        # ====================================================

        else:

            print(

                f"\n{Fore.RED}"
                "❌ Invalid choice. "
                "Enter 1, 2, 3, or 4."

            )


# ============================================================
# RUN PROGRAM
# ============================================================

if __name__ == "__main__":

    main()